# Find Chinese influencers on Douyin and Xiaohongshu, with their official rates

Search Douyin's **Xingtu** and Xiaohongshu's **PGY** creator marketplaces by topic and follower range, and get one table: followers, content categories, median views and engagements, every quoted rate in yuan, and what 1,000 median views cost. No Chinese business account, Douyin or Xiaohongshu login needed.

The script is [`kolfinder.py`](https://github.com/okdataapi/china-kol-finder) (standard library only). It calls the [okdata](https://okdata.online/?src=colab-china-kol-finder) API: a page of about 20 Douyin creators is $0.004, a page of about 20 Xiaohongshu creators $0.04.

中文：按内容方向和粉丝区间同时搜抖音星图和小红书蒲公英，得到达人报价和千次播放成本。把下面的 `QUERY` 换成你的品类。

## 1. Your API key

Get a key at [okdata.online](https://okdata.online/signup/?src=colab-china-kol-finder) (free trial: up to 333 calls). In Colab, the easiest way is the key icon in the left sidebar (Secrets): add `OKDATA_API_KEY` and allow this notebook to read it. Otherwise the cell below asks for it. The key is kept in this session only.

In [ ]:
import os, getpass

def find_key():
    try:  # Colab secrets
        from google.colab import userdata
        return userdata.get("OKDATA_API_KEY")
    except Exception:
        pass
    try:  # Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("OKDATA_API_KEY")
    except Exception:
        pass
    return None

if not os.environ.get("OKDATA_API_KEY"):
    os.environ["OKDATA_API_KEY"] = find_key() or getpass.getpass("okdata API key (okd_...): ")
print("key loaded" if os.environ["OKDATA_API_KEY"].startswith("okd_") else "that does not look like an okdata key")

## 2. Get the script

In [ ]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/okdataapi/china-kol-finder/main/kolfinder.py", "kolfinder.py")
print("kolfinder.py downloaded")

## 3. Choose the topic

Search in Chinese: creators describe their content in Chinese, so 护肤 (skincare) finds far more than "skincare". Some starting points: 露营 camping, 护肤 skincare, 美妆 makeup, 母婴 baby, 健身 fitness, 咖啡 coffee, 宠物 pets, 数码 gadgets. `--dry-run` shows the plan and the most it can cost before anything is spent.

In [ ]:
import subprocess, sys

QUERY = "露营"            # camping
FOLLOWERS = "100k-1m"     # 100k-1m, 100k- (at least) or -500k (at most)
PLATFORMS = ["douyin", "xiaohongshu"]
PAGES = 1                 # pages per platform, about 20 creators each
CNY_PER_USD = "7.1"       # use the day's rate

args = ["--query", QUERY, "--followers", FOLLOWERS, "--pages", str(PAGES), "--cny-per-usd", CNY_PER_USD,
        "--out", "shortlist.csv"] + [a for p in PLATFORMS for a in ("--platform", p)]

def run(*extra):
    r = subprocess.run([sys.executable, "kolfinder.py", *args, *extra], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)

run("--dry-run")

## 4. Search

In [ ]:
run()

## 5. The shortlist

Compare creators within a platform: Douyin counts video plays, Xiaohongshu counts note reads. Check `categories` before you shortlist anyone; a content search matches loosely.

In [ ]:
import pandas as pd

df = pd.read_csv("shortlist.csv")
cols = ["platform", "name", "followers", "categories", "median_views", "cheapest_rate", "cheapest_rate_cny",
        "cny_per_1k_views", "usd_per_1k_views", "marketplace_url"]
df.sort_values(["platform", "cny_per_1k_views"]).groupby("platform").head(10)[cols]  # 10 best value per platform

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.5))
for y, (platform, colour) in enumerate([("xiaohongshu", "#eb6834"), ("douyin", "#2a78d6")]):
    v = df.loc[df.platform == platform, "cny_per_1k_views"].dropna()
    ax.scatter(v, [y] * len(v), s=60, color=colour, edgecolor="white", linewidth=1.5, zorder=3)
    if len(v):
        ax.axvline(v.median(), ymin=y / 2 + 0.05, ymax=y / 2 + 0.45, color="black", linewidth=2)
ax.set_xscale("log")
ax.set_yticks([0, 1], ["Xiaohongshu\n(per 1k reads)", "Douyin\n(per 1k plays)"])
ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("yuan per 1,000 median views at the cheapest rate (log scale); black line = median")
ax.grid(axis="x", color="#e4e3df")
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout()
plt.show()

## 6. Next steps

Download `shortlist.csv` from the file panel. The same key also reads each creator's audience (gender, age, city, interests) and recent posts, and the same marketplace searches work from Claude or Cursor through okdata's MCP server: see [okdata.online/mcp](https://okdata.online/mcp/?src=colab-china-kol-finder). The code is on [GitHub](https://github.com/okdataapi/china-kol-finder) under the MIT license.